# AI Sentiment Intelligence
## AI-Based Sentiment Analysis System Using Machine Learning
**Academic Minor Project** | Final Deadline: 11 September 2026

### STAGE 2: DATASET + NLP PREPROCESSING ENGINE
This notebook implements an end-to-end data engineering and Natural Language Processing (NLP) preprocessing pipeline that transforms raw sentiment text into high-quality, normalized tokens suitable for TF-IDF feature extraction and machine learning classifiers in Stage 3.

**Target Polarities:**
* **Positive**
* **Negative**
* **Neutral**


## 1. Import Libraries
Load core scientific, visualization, and custom modular source packages.


In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path for modular imports
sys.path.insert(0, os.path.abspath('..'))

import config
from src.data_loader import (
    load_dataset,
    get_default_dataset_path,
    validate_dataset,
    format_validation_summary,
    report_dataset_info,
    calculate_text_statistics,
    plot_sentiment_distribution,
    plot_text_length_distribution,
    generate_data_quality_report,
)
from src.data_cleaning import (
    clean_text_basic,
    safe_str,
    to_lowercase,
    remove_urls,
    remove_html_tags,
    remove_emails,
    remove_special_characters,
    remove_extra_whitespace,
    clean_dataset,
)
from src.preprocessing import (
    expand_contractions,
    tokenize_text,
    get_controlled_stopwords,
    remove_stopwords,
    lemmatize_tokens,
    preprocess_text,
    preprocess_dataset,
    analyze_word_frequencies,
    plot_word_frequency_analysis,
)

# Set plotting aesthetics
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("All libraries and modular project components imported successfully!")


All libraries and modular project components imported successfully!


## 2. Project Configuration
Centralized paths, supported classes, and NLP configuration parameters.


In [2]:
config.ensure_directories()

print(f"Base Directory:        {config.BASE_DIR}")
print(f"Raw Data Path:         {config.RAW_DATA_PATH}")
print(f"Processed Data Path:   {config.PROCESSED_DATA_PATH}")
print(f"Results Path:          {config.RESULTS_PATH}")
print(f"Supported Classes:     {config.SUPPORTED_LABELS}")
print(f"Random State:          {config.RANDOM_STATE}")
print(f"Preserved Negations:   {sorted(list(config.PRESERVED_WORDS))}")


Base Directory:        C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence
Raw Data Path:         C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\raw
Processed Data Path:   C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\processed
Results Path:          C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results
Supported Classes:     ['Positive', 'Negative', 'Neutral']
Random State:          42
Preserved Negations:   ['against', 'barely', 'hardly', 'neither', 'never', 'no', 'nor', 'not', 'nothing', 'nowhere', 'scarcely', 'without']


## 3. Dataset Loading
Locates and loads the sentiment dataset.
The system automatically resolves the dataset path, prioritizing real user data if available or the Stage 2 benchmark demo dataset.


In [3]:
dataset_path = get_default_dataset_path()
print(f"Resolved Dataset Path: {dataset_path.resolve()}")

df_raw = load_dataset(dataset_path)
print(f"Successfully loaded {len(df_raw)} records.")


Resolved Dataset Path: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\raw\demo_dataset.csv
[SUCCESS] Dataset loaded successfully from: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\raw\demo_dataset.csv (60 records)
Successfully loaded 60 records.


## 4. Dataset Dimensions and Structure
Inspect total rows, columns, and data types.


In [4]:
print(f"Dataset Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print("\nColumn Data Types:")
print(df_raw.dtypes)


Dataset Shape: 60 rows x 2 columns

Column Data Types:
text         object
sentiment    object
dtype: object


## 5. Dataset Preview
Examine the first 10 records from the raw dataset.


In [5]:
df_raw.head(10)


## 6. Comprehensive Data Quality Audit
Validates schema integrity, missing values, empty strings, duplicates, and label correctness.
Saves official audit reports to `results/data_quality_report.json` and `.txt`.


In [6]:
validation_results = validate_dataset(df_raw)
print(format_validation_summary(validation_results))

# Persist quality report
generate_data_quality_report(df_raw)


DATASET VALIDATION SUMMARY
Status: VALID
Dataset Shape: 60 rows, 2 columns
Columns: ['text', 'sentiment']

Missing Values:
  Text: 0
  Sentiment: 0

Duplicate Rows: 0
Empty / Whitespace-only Strings: 0
Invalid Sentiment Label Rows: 0

Sentiment Classes:
  Positive: 20
  Negative: 20
  Neutral: 20

Text Length Statistics:
  Char Length: min=59, avg=71.6, max=82
  Word Count:  min=8, avg=11.5, max=16

Quality Check: No data integrity issues detected.
[INFO] Data quality report saved to:
  - C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\data_quality_report.json
  - C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\data_quality_report.txt


## 7. Missing Value Analysis
Check and report missing values in each feature.


In [7]:
missing = df_raw.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing cells: {missing.sum()}")


Missing values per column:
text         0
sentiment    0
dtype: int64

Total missing cells: 0


## 8. Duplicate Record Analysis
Identify redundant entries to prevent train/test data leakage and classifier bias.


In [8]:
duplicate_count = df_raw.duplicated(subset=[config.TEXT_COLUMN]).sum()
print(f"Duplicate text records detected: {duplicate_count}")


Duplicate text records detected: 0


## 9. Sentiment Class Distribution
Analyze class balance across Positive, Negative, and Neutral categories.


In [9]:
class_counts = df_raw[config.SENTIMENT_COLUMN].value_counts()
print("Sentiment Class Counts:")
print(class_counts)

# Generate publication-quality visualization
dist_plot_path = plot_sentiment_distribution(df_raw)
print(f"Saved distribution plot to: {dist_plot_path}")


Sentiment Class Counts:
sentiment
Positive    20
Negative    20
Neutral     20
Name: count, dtype: int64
[INFO] Sentiment distribution plot saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\sentiment_distribution.png
Saved distribution plot to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\sentiment_distribution.png


## 10. Text Cleaning Pipeline
Demonstration of individual cleaning functions:
- Lowercase normalization
- URL removal
- HTML tag & entity stripping
- Email removal
- Whitespace trimming


In [10]:
sample_noisy_text = "Check OUT: https://test.com! Email: help@site.com. <b>AWESOME</b> item... loved it!   "
print("Raw Sample:     ", repr(sample_noisy_text))
print("Lowercased:     ", repr(to_lowercase(sample_noisy_text)))
print("No URLs:        ", repr(remove_urls(sample_noisy_text)))
print("No HTML:        ", repr(remove_html_tags(sample_noisy_text)))
print("No Emails:      ", repr(remove_emails(sample_noisy_text)))
print("Cleaned Basic:  ", repr(clean_text_basic(sample_noisy_text)))


Raw Sample:      'Check OUT: https://test.com! Email: help@site.com. <b>AWESOME</b> item... loved it!   '
Lowercased:      'check out: https://test.com! email: help@site.com. <b>awesome</b> item... loved it!   '
No URLs:         'Check OUT:  Email: help@site.com. <b>AWESOME</b> item... loved it!   '
No HTML:         'Check OUT: https://test.com! Email: help@site.com.  AWESOME  item... loved it!   '
No Emails:       'Check OUT: https://test.com! Email: . <b>AWESOME</b> item... loved it!   '
Cleaned Basic:   'check out: email: . awesome item... loved it!'


## 11. Contraction Handling & Negation Preservation
Contractions like *don't*, *wasn't*, and *couldn't* contain vital sentiment polarity.
Expanding them to *do not*, *was not*, and *could not* ensures the negation survives stopword filtering.


In [11]:
sample_contraction = "I didn't like the battery, won't recommend it, and it isn't worth the money."
print("Before Contraction Expansion:")
print(" ", sample_contraction)
print("\nAfter Contraction Expansion:")
print(" ", expand_contractions(sample_contraction))


Before Contraction Expansion:
  I didn't like the battery, won't recommend it, and it isn't worth the money.

After Contraction Expansion:
  I did not like the battery, will not recommend it, and it is not worth the money.


## 12. Word Tokenization
Splits normalized sentences into discrete word tokens using NLTK.


In [12]:
sample_sentence = "The customer support was friendly and resolved my issue in minutes."
tokens = tokenize_text(sample_sentence)
print("Input Sentence: ", sample_sentence)
print("Tokens:         ", tokens)
print(f"Token Count:    {len(tokens)}")


Input Sentence:  The customer support was friendly and resolved my issue in minutes.
Tokens:          ['the', 'customer', 'support', 'was', 'friendly', 'and', 'resolved', 'my', 'issue', 'in', 'minutes']
Token Count:    11


## 13. Controlled Stopword Strategy
Standard stopword lists remove *not* and *no*, which inverts sentiment meaning.
Our controlled stopword list explicitly preserves all negation terms.


In [13]:
controlled_stops = get_controlled_stopwords(exclude_negations=True)
print(f"Controlled stopword count: {len(controlled_stops)}")
print(f"Is 'not' in stopword filter?    {'not' in controlled_stops} (Preserved!)")
print(f"Is 'never' in stopword filter?  {'never' in controlled_stops} (Preserved!)")
print(f"Is 'the' in stopword filter?    {'the' in controlled_stops} (Filtered)")

sample_tokens = ["this", "laptop", "is", "not", "good", "and", "never", "works"]
filtered_tokens = remove_stopwords(sample_tokens)
print("\nOriginal Tokens: ", sample_tokens)
print("Filtered Tokens: ", filtered_tokens)


Controlled stopword count: 194
Is 'not' in stopword filter?    False (Preserved!)
Is 'never' in stopword filter?  False (Preserved!)
Is 'the' in stopword filter?    True (Filtered)

Original Tokens:  ['this', 'laptop', 'is', 'not', 'good', 'and', 'never', 'works']
Filtered Tokens:  ['laptop', 'not', 'good', 'never', 'works']


## 14. WordNet Lemmatization
Morphologically reduces inflectional variants (e.g. *loved*, *loving*, *loves* -> *love*) to their dictionary base form.


In [14]:
sample_words = ["loved", "loving", "loves", "crashes", "crashing", "batteries", "delivering"]
lemmatized = lemmatize_tokens(sample_words)

for original, lemma in zip(sample_words, lemmatized):
    print(f"  {original:12} -> {lemma}")


  loved        -> love
  loving       -> love
  loves        -> love
  crashes      -> crash
  crashing     -> crash
  batteries    -> battery
  delivering   -> deliver


## 15. Before vs. After Preprocessing (Actual Dataset Samples)
Examine real examples from the dataset transformed by the complete NLP pipeline.


In [15]:
print("Genuine Dataset Samples (Before vs. After Full Preprocessing):")
print("=" * 80)
for idx, row in df_raw.head(6).iterrows():
    original = row[config.TEXT_COLUMN]
    cleaned = preprocess_text(original)
    label = row[config.SENTIMENT_COLUMN]
    print(f"[{label.upper()}]")
    print(f"  Original: {original}")
    print(f"  Cleaned:  {cleaned}\n")
print("=" * 80)


Genuine Dataset Samples (Before vs. After Full Preprocessing):
[POSITIVE]
  Original: The product quality is exceptional and exceeded all my expectations.
  Cleaned:  product quality exceptional exceed expectation

[POSITIVE]
  Original: Customer support was quick, friendly, and resolved my issue in minutes.
  Cleaned:  customer support quick friendly resolve issue minute

[POSITIVE]
  Original: I really love the clean design and smooth user experience of this app.
  Cleaned:  really love clean design smooth user experience app

[POSITIVE]
  Original: This laptop has great battery life and delivers outstanding performance.
  Cleaned:  laptop great battery life deliver outstanding performance

[POSITIVE]
  Original: The delivery arrived ahead of schedule and the packaging was in perfect condition.
  Cleaned:  delivery arrive ahead schedule package perfect condition

[POSITIVE]
  Original: Highly recommend this service to anyone looking for reliable and fast solutions.
  Cleaned:  highly

## 16. Execute Full Dataset Preprocessing
Apply the complete pipeline across all dataset records.
Raw text and original labels are preserved, and preprocessed text is stored in `clean_text`.


In [16]:
# Remove nulls/duplicates from raw data for clean processing
df_clean = df_raw.dropna(subset=[config.TEXT_COLUMN, config.SENTIMENT_COLUMN]).drop_duplicates(subset=[config.TEXT_COLUMN]).copy()

df_processed = preprocess_dataset(
    df_clean,
    text_column=config.TEXT_COLUMN,
    new_column="clean_text",
    remove_empty_cleaned=True,
)

print(f"Preprocessed DataFrame shape: {df_processed.shape}")
df_processed.head(10)


[PREPROCESSING] Running NLP pipeline on 60 records in column 'text'...
[PREPROCESSING] Completed. Preprocessed column: 'clean_text'
Preprocessed DataFrame shape: (60, 3)


## 17. Text Length & Word Count Analysis
Analyze word counts and character counts to inform TF-IDF hyperparameter selection.


In [17]:
stats = calculate_text_statistics(df_processed, text_column=config.TEXT_COLUMN)
print("Overall Character Statistics:", stats['character_stats'])
print("Overall Word Count Statistics:", stats['word_stats'])
print("\nAverage Word Counts per Sentiment Class:")
for cls, cls_stat in stats['class_breakdown'].items():
    print(f"  {cls:10}: {cls_stat['avg_words']} words/sample (count={cls_stat['count']})")

# Generate visualization
length_plot_path = plot_text_length_distribution(df_processed)
print(f"Saved text length distribution plot to: {length_plot_path}")


Overall Character Statistics: {'min': 59, 'max': 82, 'mean': 71.6, 'median': 71.0}
Overall Word Count Statistics: {'min': 8, 'max': 16, 'mean': 11.5, 'median': 11.0}

Average Word Counts per Sentiment Class:
  Positive  : 11.65 words/sample (count=20)
  Negative  : 11.65 words/sample (count=20)
  Neutral   : 11.2 words/sample (count=20)
[INFO] Text length distribution plot saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\text_length_distribution.png
Saved text length distribution plot to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\text_length_distribution.png


## 18. Word Frequency Analysis per Sentiment Class
Extract and visualize top vocabulary tokens characteristic of each polarity.


In [18]:
freq_data = analyze_word_frequencies(
    df_processed,
    text_column="clean_text",
    sentiment_column=config.SENTIMENT_COLUMN,
    top_n=10,
    save_json_path=config.WORD_FREQ_JSON,
)

freq_plot_path = plot_word_frequency_analysis(freq_data, output_path=config.WORD_FREQ_PLOT)
print(f"Saved word frequency plot to: {freq_plot_path}")
for label, words in freq_data.items():
    print(f"  {label}: {list(words.keys())[:6]}")


[INFO] Word frequencies saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\word_frequencies.json
[INFO] Word frequency plot saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\word_frequency_analysis.png
Saved word frequency plot to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\results\word_frequency_analysis.png
  Positive: ['quality', 'customer', 'friendly', 'minute', 'user', 'great']
  Negative: ['not', 'completely', 'customer', 'service', 'take', 'day']
  Neutral: ['schedule', 'take', 'package', 'afternoon', 'standard', 'cm']


## 19. Save Preprocessed Dataset
Persists the final processed data to `data/processed/cleaned_dataset.csv`.
Columns: `text` (original), `sentiment` (original), `clean_text` (preprocessed).


In [19]:
output_df = df_processed[[config.TEXT_COLUMN, config.SENTIMENT_COLUMN, "clean_text"]]
output_df.to_csv(config.CLEANED_DATA_FILE, index=False)
print(f"[SUCCESS] Cleaned dataset saved to: {config.CLEANED_DATA_FILE.resolve()}")
print(f"Total records saved: {len(output_df)}")

# Verify raw dataset remains untouched
print(f"Raw dataset file exists: {dataset_path.exists()} (UNTOUCHED)")


[SUCCESS] Cleaned dataset saved to: C:\Users\kavin\OneDrive\Desktop\Sentiment Analysis System\AI-Sentiment-Intelligence\data\processed\cleaned_dataset.csv
Total records saved: 60
Raw dataset file exists: True (UNTOUCHED)


## 20. Stage 2 Summary & Transition to Stage 3
### Stage 2 Accomplishments:
1. **Dataset Ingestion & Validation**: Verified schema, zero missing values, zero duplicates.
2. **Text Cleaning**: Safely normalized casing, stripped URLs, HTML tags, and email noise.
3. **Contraction Handling**: Expanded 50+ common English contractions (e.g., *don't* -> *do not*).
4. **Tokenization & Controlled Stopwords**: Tokenized text while strictly preserving negation signals (*not*, *never*, *no*).
5. **Morphological Lemmatization**: Unified inflectional word forms (*loved* -> *love*).
6. **Artifact Generation**: Generated `cleaned_dataset.csv`, `data_quality_report.json`, and 3 publication-ready visualizations.

### Ready for Stage 3:
- **TF-IDF Feature Extraction**: Converting `clean_text` into numerical feature matrices.
- **Model Training**: Logistic Regression, Multinomial Naive Bayes, and Linear Support Vector Machines (SVM).


---
## Stage 3: TF-IDF Feature Extraction & Model Training (Next Stage)
*To be implemented in Stage 3.*
- `TfidfVectorizer(max_features=5000, ngram_range=(1, 2))`
- Train-Test Split (80% Train, 20% Test, Stratified)
- Logistic Regression Classifier
- Multinomial Naive Bayes Classifier
- Linear Support Vector Classifier (LinearSVC)


## Stage 4: Model Evaluation & Comparison (Upcoming)
*To be generated after model training in Stage 3.*
- Accuracy, Precision, Recall, F1-Score
- Confusion Matrix Heatmaps
- Qualitative Error Analysis
- Best Model Selection


## Stage 5: Inference Engine & Model Serialization (Upcoming)
*To be implemented in Stage 5.*
- Single and batch text prediction
- Confidence / probability score breakdown
- Model & vectorizer serialization via Joblib


## Final Conclusion
*To be completed after final evaluation.*
